# Support Vector Regression
- draws a tube, ignoring minor errors, and only considers data outside the tube
- since random forest regression did worse than baseline linear regression, the data might have a linear shape
  - so will try linear kernal SVR
- works well with high dimensional data - my dataset has lots of features
- linear kernal SVR - used when data shows a linear trend, many features, interpretability is important

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

#import from the data_splitter Python script to load and split the data - prevents having to copy and paste code
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..'))) #used to move to the parent folder
from src.data_splitter import load_data, split

#load dataset and variables
df, X, y = load_data()

In [3]:
#split data by groups
X_train, X_val, X_test, y_train, y_val, y_test = split(df, X, y)

Training: 10661 rows | 70 unique players
Validation: 2169 rows | 15 unique players
Testing: 2170 rows | 15 unique players


In [4]:
#need to use one hot encoding to encode movement_type because it's a string
categorical = ["movement_type"]
numeric = [col for col in X_train.columns if col not in categorical]

#preprocess data: using StandardScaler to scale because need to calculate distances for the tube and don't want large quantities to dominate 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ]
)

#will first create linear kernal to see how it does
'''
- C is error penalty strength
- epsilon defines tolerance margin
'''
svr_linear = SVR(kernel='linear', C=100, epsilon=0.1)

#combine preprocessor and svr linear model in a pipeline
svr_linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", svr_linear)
    ]
)

#only fit pipeline on training data
svr_linear_pipeline.fit(X_train, y_train)
print("SVR Linear pipeline fitted successfully on training data")

SVR Linear pipeline fitted successfully on training data


In [6]:
#evaluation on svr model using validation data
y_val_pred = svr_linear_pipeline.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== SVR LINEAR MODEL METRICS ===")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

=== SVR LINEAR MODEL METRICS ===
MAE  : 3.2030
RMSE : 4.0348
R²   : 0.6649


In [9]:
#test withnon linear SVR 
#create nonlinear model
'''
- gamma controls influence range of data points
'''
svr_nonlinear = SVR(kernel='rbf', C=100, gamma=0.5, epsilon=0.1)

#combine preprocessor and svr nonlinear model in a pipeline
svr_nonlinear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", svr_nonlinear)
    ]
)

#only fit pipeline on training data
svr_nonlinear_pipeline.fit(X_train, y_train)
print("SVR Nonlinear pipeline fitted successfully on training data")

SVR Nonlinear pipeline fitted successfully on training data


In [11]:
#evaluation on nonlinear svr model using validation data
nonlinear_y_val_pred = svr_nonlinear_pipeline.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
nonlinear_val_mae = mean_absolute_error(y_val, nonlinear_y_val_pred)
nonlinear_val_rmse = root_mean_squared_error(y_val, nonlinear_y_val_pred)
nonlinear_val_r2 = r2_score(y_val, nonlinear_y_val_pred)

#performance summary
print("=== SVR NONLINEAR MODEL METRICS ===")
print(f"MAE  : {nonlinear_val_mae:.4f}")
print(f"RMSE : {nonlinear_val_rmse:.4f}")
print(f"R²   : {nonlinear_val_r2:.4f}")

=== SVR NONLINEAR MODEL METRICS ===
MAE  : 5.5082
RMSE : 6.9307
R²   : 0.0112


In [5]:
#since linear svr did better, try different values of C (error penalty strength) to try to do better
c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

#loop through the different c values to see which one is best
for c in c_values:
    svr_linear_c = SVR(kernel='linear', C=c, epsilon=0.1)
    svr_pipeline_c = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", svr_linear_c)
        ]
    )

    #train and compare the depths
    svr_pipeline_c.fit(X_train, y_train)
    pred = svr_pipeline_c.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = root_mean_squared_error(y_val, pred)
    r2 = r2_score(y_val, pred)
    
    print(f"C value: {str(c):<5}| MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

C value: 0.001| MAE: 3.5509 | RMSE: 4.4789 | R²: 0.5870
C value: 0.01 | MAE: 3.2096 | RMSE: 4.0423 | R²: 0.6636
C value: 0.1  | MAE: 3.2030 | RMSE: 4.0346 | R²: 0.6649
C value: 1.0  | MAE: 3.2030 | RMSE: 4.0347 | R²: 0.6649
C value: 10.0 | MAE: 3.2030 | RMSE: 4.0347 | R²: 0.6649
C value: 100.0| MAE: 3.2030 | RMSE: 4.0348 | R²: 0.6649


In [8]:
#metrics plataeu at c=.1 so will use that in final model
svr_final = SVR(kernel='linear', C=.1, epsilon=0.1)

#combine preprocessor and svr linear model in a pipeline
svr_final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", svr_final)
    ]
)

#only fit pipeline on training data
svr_final_pipeline.fit(X_train, y_train)

#evaluation on final svr model
final_y_val_pred = svr_final_pipeline.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
final_val_mae = mean_absolute_error(y_val, final_y_val_pred)
final_val_rmse = root_mean_squared_error(y_val, final_y_val_pred)
final_val_r2 = r2_score(y_val, final_y_val_pred)

#performance summary
print("=== SVR FINAL MODEL METRICS ===")
print(f"MAE  : {final_val_mae:.4f}")
print(f"RMSE : {final_val_rmse:.4f}")
print(f"R²   : {final_val_r2:.4f}")

=== SVR FINAL MODEL METRICS ===
MAE  : 3.2030
RMSE : 4.0346
R²   : 0.6649
